# Regrid a full ICON-CH1-EPS field to the Swiss LV95 grid

This example retrieves one **full ICON-CH1-EPS field** from MeteoSwiss Polytope and interpolates it to the predefined **Swiss LV95 1-km grid**.


## Environment setup

Create a clean Python 3.11 environment:

```bash
python3.11 -m venv .venv-swisslv95
source .venv-swisslv95/bin/activate

python -m pip install --upgrade pip setuptools wheel

python -m pip install \
    "earthkit-data[polytope]==1.1.0" \
    earthkit-geo==1.1.2 \
    eccodes-cosmo-resources-python==2.47.0.1 \
    ipykernel
```

Install the newer ECMWF regridding backend wheels:

```bash
python -m pip install \
    -i https://get-test.ecmwf.int/repository/pypi-betas/simple/ \
    --extra-index-url https://pypi.org/simple \
    eckit==2.3.1.dev213 \
    mir-python==1.30.2.dev213
```

Register the environment as a Jupyter kernel:

```bash
python -m ipykernel install --user \
    --name swisslv95 \
    --display-name "Python (Swiss LV95)"
```


## Configure Polytope access

To access ICON data via MeteoSwiss Polytope, create a `config.yml` based on the repository's `config_example.yml` and add your MeteoSwiss offline token.


In [ ]:
import os
import yaml

def load_config(path="config.yml"):
    if not os.path.exists(path):
        raise FileNotFoundError(
            "Missing config.yml. Please create one based on config_example.yml."
        )
    with open(path, "r") as f:
        return yaml.safe_load(f)

config = load_config()

os.environ["POLYTOPE_USER_KEY"] = config["meteoswiss"]["key"]
os.environ["POLYTOPE_ADDRESS"] = "https://service.meteoswiss.ch/polytope"


## Configure MeteoSwiss ecCodes definitions

MeteoSwiss uses additional ecCodes definitions for ICON data. Configure them before reading the GRIB field.


In [ ]:
import os

os.environ["ECCODES_ECKIT_GEO"] = "1"
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

import eccodes
import eccodes_cosmo_resources

vendor = eccodes.codes_definition_path()
cosmo = eccodes_cosmo_resources.get_definitions_path()
eccodes.codes_set_definitions_path(f"{cosmo}:{vendor}")


## Choose a recent model run automatically

Use a time approximately 12 hours in the past and round it down to the previous 3-hour model cycle. This avoids having to edit the date manually each time the notebook is run.


In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
past_time = now - timedelta(hours=12)

rounded_hour = (past_time.hour // 3) * 3
rounded_time = past_time.replace(
    hour=rounded_hour,
    minute=0,
    second=0,
    microsecond=0,
)

request_date = rounded_time.strftime("%Y%m%d")
request_time = rounded_time.strftime("%H%M")

print("Model cycle:", request_date, request_time)


## Retrieve one full ICON-CH1-EPS field

The `mch` collection returns the full field. Here we request 2-metre temperature (`T_2M`) as a simple example.


In [ ]:
import earthkit.data as ekd

request = {
    "param": "500011",       # T_2M
    "date": request_date,
    "time": request_time,
    "expver": "0001",
    "step": 0,
    "class": "od",
    "levtype": "sfc",
    "model": "icon-ch1-eps",
    "stream": "enfo",
    "type": "CF",
    "timespan": "none",
}

ds = ekd.from_source(
    "polytope",
    "mch",
    request,
    stream=False,
)

field = ds.to_fieldlist()[0]
field


## Inspect the native ICON grid


In [ ]:
print(field.ls())
print("gridType:", field.metadata("gridType"))
print("grid spec:", field.geography.grid_spec())

values = field.to_numpy()
print("number of source points:", values.size)


## Swiss LV95 target grid

**LV95** is Switzerland's national projected coordinate system, **CH1903+ / LV95 (EPSG:2056)**.

Unlike latitude/longitude coordinates, LV95 uses projected **eastings and northings in metres**. The predefined `swisslv95` grid is a regular 1-km grid covering Switzerland.

For the standard MeteoSwiss grid, the easiest form is simply:

```python
Grid("swisslv95")
```

Its 1-km grid corresponds to the following explicit specification:

```python
{
    "type": "swisslv95",
    "x": [2439000, 2867000, 1000],
    "y": [1040500, 1333500, 1000],
}
```

The explicit form is useful when a different LV95 extent or spacing is required.


In [ ]:
from eckit.geo import Grid

target_grid = Grid("swisslv95")

print("Swiss LV95 grid specification:")
print(target_grid.spec)
print("shape:", target_grid.shape)
print("size:", target_grid.size())


## Regrid ICON-CH1-EPS to Swiss LV95

`earthkit-geo` delegates the interpolation to MIR. Here we use linear interpolation.

The first interpolation for a given source grid, target grid and interpolation method may take longer because MIR computes the interpolation coefficients (weights). MIR can cache these weights and reuse them for later fields on the same pair of grids.


In [ ]:
import earthkit.geo as ekg

result, _ = ekg.grids.array.regrid(
    values,
    in_grid=field.geography.grid_spec(),
    out_grid=target_grid,
    interpolation="linear",
)

print("source points:", values.size)
print("target points:", target_grid.size())
print("output points:", result.size)


### Alternative: define the LV95 target explicitly

Instead of the predefined named grid, pass a grid specification directly:

```python
custom_target_grid = {
    "type": "swisslv95",
    "x": [2439000, 2867000, 1000],
    "y": [1040500, 1333500, 1000],
}

result, _ = ekg.grids.array.regrid(
    values,
    in_grid=field.geography.grid_spec(),
    out_grid=custom_target_grid,
    interpolation="linear",
)
```

Change the x/y bounds or spacing when a different LV95 target domain is needed.


## Sanity checks

The source ICON grid contains many more points than the 1-km Swiss target grid. The exact values depend on the selected field and model cycle.


In [ ]:
import numpy as np

print(f"Source points: {values.size}")
print(f"Target points: {target_grid.size()}")
print(f"Output points: {result.size}")

print(
    f"Source range: {np.nanmin(values):.3f} to {np.nanmax(values):.3f}"
)
print(
    f"Result range: {np.nanmin(result):.3f} to {np.nanmax(result):.3f}"
)

print(f"Source NaNs: {np.isnan(values).sum()}")
print(f"Result NaNs: {np.isnan(result).sum()}")

assert result.size == target_grid.size()


## Notes on interpolation weights

When earthkit-geo performs the regridding, it uses MIR (Meteorological Interpolation and Regridding), ECMWF's interpolation library, under the hood.

For each combination of:
- source grid,
- target grid,
- interpolation method.

MIR computes interpolation coefficients. These coefficients can be cached and reused for repeated interpolations on the same grids.

The cache location depends on the system configuration. On some systems it may use a temporary directory such as /tmp, so the cache may be recreated when the environment or machine changes.